# Interpolação de Lagrange — Derivação passo a passo

Dados os pontos do enunciado: $x_0=-1$, $x_1=0$, $x_2=2$ e $f(x_0)=4$, $f(x_1)=1$, $f(x_2)=-1$.

Definimos os polinômios base de Lagrange de grau 2:

$$
l_0(x) = \frac{(x-x_1)(x-x_2)}{(x_0-x_1)(x_0-x_2)},\quad
l_1(x) = \frac{(x-x_0)(x-x_2)}{(x_1-x_0)(x_1-x_2)},\quad
l_2(x) = \frac{(x-x_0)(x-x_1)}{(x_2-x_0)(x_2-x_1)}.
$$

Substituindo $x_0=-1$, $x_1=0$, $x_2=2$:

- $l_0(x)$:
  $$
  l_0(x) = \frac{(x-0)(x-2)}{(-1-0)(-1-2)} = \frac{x(x-2)}{(-1)(-3)} = \frac{x^2 - 2x}{3}.
  $$
- $l_1(x)$:
  $$
  l_1(x) = \frac{(x-(-1))(x-2)}{(0-(-1))(0-2)} = \frac{(x+1)(x-2)}{(1)(-2)} = \frac{x^2 - x - 2}{-2}.
  $$
- $l_2(x)$:
  $$
  l_2(x) = \frac{(x-(-1))(x-0)}{(2-(-1))(2-0)} = \frac{(x+1)x}{(3)(2)} = \frac{x^2 + x}{6}.
  $$

Portanto, o polinômio interpolador de grau 2 é
$$
P_2(x) = f_0\,l_0(x) + f_1\,l_1(x) + f_2\,l_2(x).
$$

Com $f_0=4$, $f_1=1$, $f_2=-1$:
$$
\begin{aligned}
P_2(x)
&= 4\left(\frac{x^2 - 2x}{3}\right) + 1\left(\frac{x^2 - x - 2}{-2}\right) - 1\left(\frac{x^2 + x}{6}\right) \\
&= \frac{4}{3}(x^2 - 2x) - \frac{1}{2}(x^2 - x - 2) - \frac{1}{6}(x^2 + x) \\
&= \left(\frac{4}{3} - \frac{1}{2} - \frac{1}{6}\right)x^2 + \left(-\frac{8}{3} + \frac{1}{2} - \frac{1}{6}\right)x + \left(\frac{1}{1}\cdot\frac{1}{1} \times 1\right) \\
&= \frac{2}{3}x^2 - \frac{7}{3}x + 1.
\end{aligned}
$$

Verificações esperadas: $P_2(x_0)=4$, $P_2(x_1)=1$, $P_2(x_2)=-1$; e a forma expandida deve coincidir com $P_2(x) = \frac{2}{3}x^2 - \frac{7}{3}x + 1$.


In [2]:
# Interpolação de Lagrange genérica (frações + numérico)
# Funciona para qualquer conjunto de pontos distintos x, y.
import numpy as np
from fractions import Fraction
from typing import List, Tuple

# ========== Utilidades de polinômios com Fractions (coeficientes em ordem crescente) ==========

def poly_add(p: List[Fraction], q: List[Fraction]) -> List[Fraction]:
    m = max(len(p), len(q))
    r = [Fraction(0) for _ in range(m)]
    for i in range(m):
        if i < len(p):
            r[i] += p[i]
        if i < len(q):
            r[i] += q[i]
    return r

def poly_scale(p: List[Fraction], s: Fraction) -> List[Fraction]:
    return [coef * s for coef in p]

def poly_mul(p: List[Fraction], q: List[Fraction]) -> List[Fraction]:
    r = [Fraction(0) for _ in range(len(p) + len(q) - 1)]
    for i, a in enumerate(p):
        for j, b in enumerate(q):
            r[i + j] += a * b
    return r

# x -> representado por [0,1], constante c -> [c]
X_poly = [Fraction(0), Fraction(1)]  # x

# ========== Lagrange genérico com Fractions ==========

def lagrange_fraction(xs: List[float], ys: List[float]) -> List[Fraction]:
    """
    Constrói o polinômio interpolador via Lagrange para pontos (xs, ys).
    Retorna lista de coeficientes Fraction em ordem crescente (a0, a1, ..., a_n)
    Pode receber xs/ys como int/float; converte para Fraction.
    """
    if len(xs) != len(ys):
        raise ValueError("xs e ys devem ter mesmo tamanho.")
    n = len(xs)
    if n == 0:
        raise ValueError("Forneça ao menos um ponto.")
    # Verifica pontos distintos
    if len(set(xs)) != n:
        raise ValueError("Todos os x devem ser distintos.")
    xsF = [Fraction(x).limit_denominator() for x in xs]
    ysF = [Fraction(y).limit_denominator() for y in ys]
    P = [Fraction(0)]  # polinômio acumulado
    for k in range(n):
        # Numerador do l_k(x): produto (x - x_j), j != k
        numer = [Fraction(1)]  # polinômio constante 1
        denom = Fraction(1)
        for j in range(n):
            if j != k:
                # (x - x_j) = X_poly - [x_j]
                numer = poly_mul(numer, poly_add(X_poly, [ -xsF[j] ]))
                denom *= (xsF[k] - xsF[j])
        lk = poly_scale(numer, Fraction(1,1)/denom)  # divide pelo denominador
        # y_k * l_k(x)
        term = poly_scale(lk, ysF[k])
        P = poly_add(P, term)
    return P  # coeficientes em ordem crescente

# ========== Versão numérica genérica com numpy (qualquer N) ==========

def lagrange_numeric(xs: np.ndarray, ys: np.ndarray) -> np.poly1d:
    p = np.poly1d([0.0])
    Xsym = np.poly1d([1.0, 0.0])
    n = len(xs)
    for k in range(n):
        lk = np.poly1d([1.0])
        den = 1.0
        for j in range(n):
            if j != k:
                lk *= (Xsym - xs[j])
                den *= (xs[k] - xs[j])
        lk /= den
        p += ys[k] * lk
    return p  # coeficientes em ordem decrescente

# ========== Exemplo: 5 pontos da questão (gera polinômio de grau ≤4, mas sairá grau 3) ==========
x = np.array([-2, 0, 1, 3], dtype=float)
y = np.array([-27, -1, 6, 128], dtype=float)


# Polinômio exato via Fractions
P_frac = lagrange_fraction(list(x), list(y))  # [a0, a1, ..., a_n]
# Remove coeficientes de grau mais alto que sejam zero (limpeza estética)
while len(P_frac) > 1 and P_frac[-1] == 0:
    P_frac.pop()

# Polinômio numérico (numpy) para avaliação
P_num = lagrange_numeric(x, y)

print("=== Interpolação de Lagrange Genérica ===")
print(f"Pontos x: {x.tolist()}")
print(f"Valores y: {y.tolist()}")
print("\nCoeficientes exatos (Fraction, ordem crescente):")
print(P_frac)  # mostra lista

# Monta string simbólica com frações
terms = []
for deg, coef in enumerate(P_frac):
    if coef == 0:
        continue
    if deg == 0:
        terms.append(f"{coef}")
    elif deg == 1:
        terms.append(f"({coef})x")
    else:
        terms.append(f"({coef})x^{deg}")
poly_frac_str = " + ".join(terms) if terms else "0"
print("Forma exata: P(x) =", poly_frac_str)

print("\nCoeficientes numéricos (numpy, ordem decrescente):")
print(P_num.c)
print("Forma numérica: P(x) =", P_num)

# Verificação
print("\nVerificação nos nós:")
for i, xi in enumerate(x):
    vi = float(np.polyval(P_num, xi))
    err = abs(vi - y[i])
    print(f"x={xi:6.2f} -> P(x)={vi:10.6f}, y={y[i]:10.6f}, erro={err:.2e}")

# Teste em ponto intermediário
Xtest = 1.5
print(f"\nAvaliação em X={Xtest}:")
print("P(1.5) =", float(np.polyval(P_num, Xtest)))

# Nota: O resultado deve coincidir com P(x) = x^3 - 6x^2 - 4x + 3 (grau 3)
# Confirmando formatado:
# Extrai somente coeficientes relevantes (grau 3)
a_dec = list(P_num.c)  # [a3, a2, a1, a0]
if len(a_dec) >= 4:
    print("\nForma simplificada (grau 3): P(x) = {:.0f}x^3 + ({:.0f})x^2 + ({:.0f})x + ({:.0f})".format(a_dec[0], a_dec[1], a_dec[2], a_dec[3]))


=== Interpolação de Lagrange Genérica ===
Pontos x: [-2.0, 0.0, 1.0, 3.0]
Valores y: [-27.0, -1.0, 6.0, 128.0]

Coeficientes exatos (Fraction, ordem crescente):
[Fraction(-1, 1), Fraction(1, 1), Fraction(2, 1), Fraction(4, 1)]
Forma exata: P(x) = -1 + (1)x + (2)x^2 + (4)x^3

Coeficientes numéricos (numpy, ordem decrescente):
[ 4.  2.  1. -1.]
Forma numérica: P(x) =    3     2
4 x + 2 x + 1 x - 1

Verificação nos nós:
x= -2.00 -> P(x)=-27.000000, y=-27.000000, erro=0.00e+00
x=  0.00 -> P(x)= -1.000000, y= -1.000000, erro=0.00e+00
x=  1.00 -> P(x)=  6.000000, y=  6.000000, erro=0.00e+00
x=  3.00 -> P(x)=128.000000, y=128.000000, erro=0.00e+00

Avaliação em X=1.5:
P(1.5) = 18.5

Forma simplificada (grau 3): P(x) = 4x^3 + (2)x^2 + (1)x + (-1)


In [4]:
from sympy import symbols, expand

def lagrange_interpolacao_verbose(xs, ys):
    """
    Calcula o polinômio interpolador de Lagrange

    xs: lista de abscissas [x0, x1, ..., xn]
    ys: lista de ordenadas [y0, y1, ..., yn]
    """
    if len(xs) != len(ys):
        raise ValueError("As listas xs e ys precisam ter o mesmo tamanho.")
    if len(set(xs)) != len(xs):
        raise ValueError("Os x_i devem ser todos distintos para interpolação de Lagrange.")

    x = symbols('x')
    n = len(xs)

    print("="*70)
    print("INTERPOLAÇÃO DE LAGRANGE")
    print("="*70, "\n")

    print("1) Pontos de interpolação (x_i, y_i):")
    for i, (xi, yi) in enumerate(zip(xs, ys)):
        print(f"   i = {i}:  x_{i} = {xi:>6},   y_{i} = {yi}")
    print("\n2) Construindo os polinômios básicos de Lagrange L_i(x):\n")

    P = 0  # polinômio final

    for i in range(n):
        xi = xs[i]

        # Construir numerador e denominador
        num_factors = []  # fatores (x - x_j)
        den = 1           # produto (x_i - x_j)

        for j in range(n):
            if j == i:
                continue
            xj = xs[j]
            num_factors.append(x - xj)
            den *= (xi - xj)

        # Expressão simbólica do numerador: produto dos fatores
        num_expr = 1
        for factor in num_factors:
            num_expr *= factor

        Li = num_expr / den  # L_i(x)

        # Impressões detalhadas do L_i(x)
        print(f"   L_{i}(x) = Π_(j≠{i}) (x - x_j) / (x_{i} - x_j)")

        num_str = "".join([f"(x - {xs[j]})" for j in range(n) if j != i])
        den_str_parts = [f"({xi} - {xs[j]})" for j in range(n) if j != i]
        den_str = " * ".join(den_str_parts)

        print(f"          = {num_str} / {den_str}")
        print(f"          = ({expand(num_expr)}) / ({den})")
        print(f"          = {expand(Li)}\n")

        # Acumula no polinômio final: y_i * L_i(x)
        P += ys[i] * Li

    print("3) Polinômio interpolador:")
    print("   P(x) = Σ y_i * L_i(x)\n")

    for i in range(n):
        print(f"   Termo {i}:  y_{i} * L_{i}(x) = {ys[i]} * L_{i}(x)")

    print("\n4) Somando e simplificando todos os termos:")
    P_simplificado = expand(P)
    print(f"\n   P(x) = {P_simplificado}")
    print("\nFim da construção.\n" + "="*70)

    return P_simplificado


if __name__ == "__main__":
    xs = [-1, 0, 1, 2]
    ys = [0, 2, 2, 6]

    polinomio = lagrange_interpolacao_verbose(xs, ys)
    print("\nPolinômio final (forma fechada):")
    print("P(x) =", polinomio)


INTERPOLAÇÃO DE LAGRANGE

1) Pontos de interpolação (x_i, y_i):
   i = 0:  x_0 =     -1,   y_0 = 0
   i = 1:  x_1 =      0,   y_1 = 2
   i = 2:  x_2 =      1,   y_2 = 2
   i = 3:  x_3 =      2,   y_3 = 6

2) Construindo os polinômios básicos de Lagrange L_i(x):

   L_0(x) = Π_(j≠0) (x - x_j) / (x_0 - x_j)
          = (x - 0)(x - 1)(x - 2) / (-1 - 0) * (-1 - 1) * (-1 - 2)
          = (x**3 - 3*x**2 + 2*x) / (-6)
          = -x**3/6 + x**2/2 - x/3

   L_1(x) = Π_(j≠1) (x - x_j) / (x_1 - x_j)
          = (x - -1)(x - 1)(x - 2) / (0 - -1) * (0 - 1) * (0 - 2)
          = (x**3 - 2*x**2 - x + 2) / (2)
          = x**3/2 - x**2 - x/2 + 1

   L_2(x) = Π_(j≠2) (x - x_j) / (x_2 - x_j)
          = (x - -1)(x - 0)(x - 2) / (1 - -1) * (1 - 0) * (1 - 2)
          = (x**3 - x**2 - 2*x) / (-2)
          = -x**3/2 + x**2/2 + x

   L_3(x) = Π_(j≠3) (x - x_j) / (x_3 - x_j)
          = (x - -1)(x - 0)(x - 1) / (2 - -1) * (2 - 0) * (2 - 1)
          = (x**3 - x) / (6)
          = x**3/6 - x/6

3) Polinômio

In [5]:
from sympy import symbols, Rational, expand, latex, fraction
from IPython.display import display, Math


def lagrange_interpolacao_latex(xs, ys, var_name="x"):
    """
    Interpolação de Lagrange com saída em LaTeX no estilo 'resolvendo à mão'.
    
    xs: lista dos valores x_i
    ys: lista dos valores y_i
    var_name: nome da variável (ex.: 'x')
    """
    # variável simbólica
    x = symbols(var_name)

    # converte para frações exatas
    xs = [Rational(v) for v in xs]
    ys = [Rational(v) for v in ys]

    n = len(xs)
    if n != len(ys):
        raise ValueError("xs e ys devem ter o mesmo tamanho.")
    if len(set(xs)) != n:
        raise ValueError("Os x_i devem ser distintos.")

    # Mostrar pontos
    display(Math(r"\textbf{Pontos de interpolação:}"))
    for i, (xi, yi) in enumerate(zip(xs, ys)):
        display(Math(f"(x_{i}, f(x_{i})) = ({latex(xi)},\, {latex(yi)})"))

    # Mostrar fórmula geral
    display(Math(
        r"P_n(x) = \sum_{i=0}^{n} y_i L_i(x), \quad"
        r"L_i(x)=\prod_{\substack{j=0 \\ j\neq i}}^{n}\frac{x-x_j}{x_i-x_j}"
    ))

    L_terms = []
    P = 0  # polinômio final

    # Construção de cada L_i(x)
    for i in range(n):
        xi = xs[i]

        # Numerador e denominador simbólicos
        num = 1
        den = 1
        num_tex = ""
        den_tex = ""

        for j in range(n):
            if j == i:
                continue
            xj = xs[j]
            num *= (x - xj)
            num_tex += r"(x - %s)" % latex(xj)
            den *= (xi - xj)
            den_tex += r"(%s - %s)" % (latex(xi), latex(xj))

        Li = num / den
        L_terms.append(Li)

        # ==== Exibição em LaTeX ====
        display(Math(r"\textbf{Polinômio\ básico\ }L_%d(x)" % i))

        # Forma do produto
        display(Math(
            r"L_%d(x)=\frac{%s}{%s}"
            % (i, num_tex, den_tex)
        ))

        # Numerador expandido
        display(Math(
            r"\text{Numerador expandido: } %s"
            % latex(expand(num))
        ))

        # Denominador numérico
        display(Math(
            r"\text{Denominador: } %s"
            % latex(den)
        ))

        # L_i simplificado
        display(Math(
            r"L_%d(x)=\frac{%s}{%s} = %s"
            % (i, latex(expand(num)), latex(den), latex(expand(Li)))
        ))

        # Termo do polinômio
        Ti = ys[i] * Li
        num_Ti, den_Ti = fraction(Ti)
        display(Math(
            r"T_{%d}(x)=y_%d L_%d(x) = %s\cdot (%s)"
            % (i, i, i, latex(ys[i]), latex(expand(Li)))
        ))
        display(Math(
            r"T_{%d}(x)=\frac{%s}{%s}"
            % (i, latex(expand(num_Ti)), latex(den_Ti))
        ))
        display(Math(
            r"T_{%d}(x)= %s"
            % (i, latex(expand(Ti)))
        ))

        P += Ti

    # ============================
    # Polinômio final
    # ============================
    P_exp = expand(P)
    num_P, den_P = fraction(P_exp)

    display(Math(r"\textbf{Polinômio interpolador final:}"))
    display(Math(r"P(x)= %s" % latex(P_exp)))

    if den_P != 1:
        display(Math(
            r"P(x)=\dfrac{%s}{%s}"
            % (latex(expand(num_P)), latex(den_P))
        ))

    return P_exp


# EXECUÇÃO
if __name__ == "__main__":
    xs = [-1, 0, 1, 2]
    ys = [0, 2, 2, 6]
    P = lagrange_interpolacao_latex(xs, ys)
    print("P(x) =", P)


<>:29: SyntaxWarning: invalid escape sequence '\,'
<>:29: SyntaxWarning: invalid escape sequence '\,'
C:\Users\alano\AppData\Local\Temp\ipykernel_109300\2342852061.py:29: SyntaxWarning: invalid escape sequence '\,'
  display(Math(f"(x_{i}, f(x_{i})) = ({latex(xi)},\, {latex(yi)})"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

P(x) = x**3 - x**2 + 2
